[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Overfitting and Regularization


## What you will be able to do

See a network overfit in its validation loss, slow that down with dropout and weight decay, and stop
training at the best epoch, keeping that epoch's weights, in PyTorch and in Keras.

## The idea

### The problem

A network of 17,281 weights trained on 100 stations has far more weights than examples. It can
bend itself through every one of those 100 January means, including the random part that the
stations' generator added to each, which no feature explains. On the stations it trains on, its
loss keeps falling. On stations it has never seen, the bends that fit that noise make it worse, and the
validation loss, after falling with the training loss for a while, turns and rises.

Fewer weights would overfit less, and also fit less. The usual answer keeps the large network and
makes it harder for it to memorize, or stops training when the validation loss is lowest, before the
memorizing has gone far.

### A formal definition

> A model **overfits** when changes that lower its loss on the training data raise its loss on data
> it has not seen. **Regularization** is any change to a model or to how it trains that reduces
> overfitting on the same data, such as **dropout**, which sets a random share of a layer's outputs to
> 0 at every training step, and **weight decay**, which shrinks every weight a little at every step.
> **Early stopping**
> ends training once the validation loss has not improved for a set number of epochs, called the
> **patience**, and keeps the weights from the best epoch.

### Why it works that way

Dropout makes every training step use a different, thinned network, so no unit can rely on one
particular other unit being there, and fitting one station's noise through a single path stops
paying off. The surviving outputs are scaled up during training to make up for the missing ones, and
in evaluation dropout does nothing at all, so the full network answers. That is why
`model.train()` and `model.eval()` matter once a network has dropout.

Weight decay pulls every weight toward 0 in proportion to its size, and only weights that keep
lowering the loss stay large. PyTorch's `AdamW` and Keras's `AdamW` take it as `weight_decay`. Keras
can also add a penalty on the weights to the loss itself, through a layer's `kernel_regularizer`, and
the loss that `fit` and `evaluate` report then includes the penalty, which a task in Your turn shows.

Early stopping needs a copy of the best weights, because training goes on past the best epoch until
the patience runs out. In PyTorch, a model's `state_dict()` holds tensors that share memory with its
weights, so keeping the best weights means copying them, which Common errors shows.

### Where this shows up

The **Training Loops** notebook first separated the validation loss from the training loss, and
the **Saving and Loading Models** notebook saves a model's `state_dict` to a file. The
**A Complete Project** notebook trains its classifier with early stopping.

### What this notebook covers

- a large network on 100 stations, and a small one for comparison
- dropout, and what training and evaluation mode do to it
- weight decay with `AdamW` in both frameworks
- early stopping by hand in PyTorch and with Keras's `EarlyStopping`
- four mistakes, none of which raises an error, though one draws a warning

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

torch.manual_seed(0)
dropout = torch.nn.Dropout(0.5)
ones = torch.ones(8)

dropout.train()
print(dropout(ones))
dropout.eval()
print(dropout(ones))
```

```
tensor([0., 0., 2., 0., 0., 0., 2., 2.])
tensor([1., 1., 1., 1., 1., 1., 1., 1.])
```

In training mode the layer set five of its eight inputs to 0 this time, a new random choice on
every call, and doubled the rest, so that their sum stays about the same on average. In evaluation
mode it passes everything through unchanged.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, two tools for Common errors, and the
stations.

- `numpy` builds the stations, and finds the best epoch in a record of losses
- `torch` is PyTorch, whose `nn.Dropout` and `optim.AdamW` this notebook uses
- `tensorflow` is TensorFlow, and `keras` its library of layers, optimizers and callbacks
- `contextlib` and `os` hide the messages TensorFlow prints while it loads
- `copy` makes a deep copy of a model's weights
- `warnings` catches the warning in the last of the Common errors, so that it prints once

TensorFlow's C++ code prints several lines as it loads, each stamped with the time and a process
number, so no two runs would show the same output. Setting `TF_CPP_MIN_LOG_LEVEL` to 3 hides what
TensorFlow logs once it has started, and `c_messages_hidden` sends the lines it prints while it
starts to `os.devnull`. The cell prints PyTorch's version without the part after `+`, which names
the build.

`station_network_arrays` makes the 1,200 made-up weather stations of the **scikit-learn, Deep Dive**
guide, drawn from a fixed seed, so they are the same on every run. `features` holds each station's
latitude, longitude, elevation and distance from the coast as 32-bit floats, and `jan_mean` its
January mean in degrees Celsius. The cell splits the stations into 800 for training, 200 for
validation and 200 for testing, and standardizes the features with the training stations' means
into `scaled`. This notebook trains on only the first 100 training stations, so that a network can
overfit them quickly.

In [1]:
import contextlib
import copy
import os
import warnings

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


## Worked examples

### A large network and 100 stations

`large_network` builds two hidden layers of 128 units, with a dropout layer after each when it is
given a rate.

In [2]:
def as_tensors(rows):
    """The standardized features of these rows, and their January means as a column."""
    return torch.from_numpy(scaled[rows]), torch.from_numpy(jan_mean[rows]).unsqueeze(1)


few_rows = train_rows[:100]
X_few, y_few = as_tensors(few_rows)
X_valid, y_valid = as_tensors(valid_rows)
X_test, y_test = as_tensors(test_rows)
mse = torch.nn.MSELoss()


def large_network(dropout=0.0):
    """A 4-128-128-1 network, with dropout after each hidden layer when dropout is above 0."""
    layers = [torch.nn.Linear(4, 128), torch.nn.ReLU()]
    if dropout:
        layers.append(torch.nn.Dropout(dropout))
    layers += [torch.nn.Linear(128, 128), torch.nn.ReLU()]
    if dropout:
        layers.append(torch.nn.Dropout(dropout))
    layers.append(torch.nn.Linear(128, 1))
    return torch.nn.Sequential(*layers)


print(sum(p.numel() for p in large_network().parameters()), "weights for", len(few_rows), "training stations")

17281 weights for 100 training stations


### The validation loss turns while the training loss falls

`fit_full_batch` takes one step on all 100 stations per epoch and records both losses in evaluation
mode. `show` prints every fiftieth epoch and the best one.

In [3]:
def fit_full_batch(model, optimizer, epochs=400):
    """One step per epoch on the 100 stations; returns the training and validation loss of every epoch."""
    history = {"loss": [], "val_loss": []}
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        mse(model(X_few), y_few).backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            history["loss"].append(mse(model(X_few), y_few).item())
            history["val_loss"].append(mse(model(X_valid), y_valid).item())
    return history


def show(history, every=50):
    """Print both losses every few epochs, then the epoch with the lowest validation loss."""
    for epoch in range(every, len(history["loss"]) + 1, every):
        print(f"epoch {epoch:3}: training loss {history['loss'][epoch - 1]:.3f}, "
              f"validation loss {history['val_loss'][epoch - 1]:.3f}")
    best = int(np.argmin(history["val_loss"]))
    print(f"lowest validation loss at epoch {best + 1}: {history['val_loss'][best]:.3f}")


torch.manual_seed(0)
large = large_network()
show(fit_full_batch(large, torch.optim.Adam(large.parameters(), lr=0.01)))

epoch  50: training loss 0.798, validation loss 0.917
epoch 100: training loss 0.508, validation loss 0.889
epoch 150: training loss 0.393, validation loss 0.986
epoch 200: training loss 0.315, validation loss 1.082
epoch 250: training loss 0.262, validation loss 1.153
epoch 300: training loss 0.226, validation loss 1.211
epoch 350: training loss 0.210, validation loss 1.355
epoch 400: training loss 0.172, validation loss 1.330
lowest validation loss at epoch 76: 0.840


The training loss falls at every printed epoch, from 0.798 to 0.172. The validation loss is lowest
at epoch 76, at 0.84, and from there it climbs, to 1.355 at epoch 350 and 1.33 at epoch 400, as the
network fits the 100 stations more and more closely and the other stations less well.

### A small network for comparison

This network has eight hidden units and 49 weights, and trains the same way.

In [4]:
torch.manual_seed(0)
small = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.ReLU(), torch.nn.Linear(8, 1))
print(sum(p.numel() for p in small.parameters()), "weights")
show(fit_full_batch(small, torch.optim.Adam(small.parameters(), lr=0.01)), every=100)

49 weights
epoch 100: training loss 2.836, validation loss 2.870
epoch 200: training loss 1.589, validation loss 1.553
epoch 300: training loss 1.235, validation loss 1.200
epoch 400: training loss 1.176, validation loss 1.145
lowest validation loss at epoch 400: 1.145


At every printed epoch the small network's two losses are within 0.04 of each other, and both are
still falling at epoch 400, at 1.176 and 1.145, well above the large network's best of 0.84.

### Dropout

This is the large network again, with a dropout rate of 0.3 after each hidden layer.

In [5]:
torch.manual_seed(0)
dropout_net = large_network(dropout=0.3)
show(fit_full_batch(dropout_net, torch.optim.Adam(dropout_net.parameters(), lr=0.01)))

epoch  50: training loss 1.028, validation loss 1.063
epoch 100: training loss 0.721, validation loss 0.903
epoch 150: training loss 0.664, validation loss 0.859
epoch 200: training loss 0.566, validation loss 0.867
epoch 250: training loss 0.571, validation loss 0.807
epoch 300: training loss 0.549, validation loss 0.951
epoch 350: training loss 0.513, validation loss 0.824
epoch 400: training loss 0.475, validation loss 0.877
lowest validation loss at epoch 290: 0.787


With dropout, the training loss falls more slowly, to 0.475 by epoch 400 against 0.172. From epoch
100 on, the printed validation losses stay between 0.807 and 0.951 instead of climbing past 1.3. The
lowest, 0.787 at epoch 290, is below the large network's 0.84 in this run, which comes from one
seed, and the Keras runs below put dropout's lowest point above the plain network's.

### Dropout in training and in evaluation mode

The cell asks the trained network for one validation station's prediction three times in each mode.

In [6]:
station = X_valid[:1]
with torch.no_grad():
    dropout_net.train()
    print("training mode:  ", [round(dropout_net(station).item(), 2) for _ in range(3)])
    dropout_net.eval()
    print("evaluation mode:", [round(dropout_net(station).item(), 2) for _ in range(3)])

training mode:   [-1.97, -1.09, -1.26]
evaluation mode: [-1.55, -1.55, -1.55]


In training mode each call drops a different set of units and gives a different answer, from −1.97
to −1.09 here. In evaluation mode every unit takes part, and the answer is the same every time.

### Weight decay

`AdamW` with `weight_decay=1.0` shrinks every weight by the learning rate times the decay, 1%, at
every step, on top of Adam's step.

In [7]:
torch.manual_seed(0)
decayed = large_network()
show(fit_full_batch(decayed, torch.optim.AdamW(decayed.parameters(), lr=0.01, weight_decay=1.0)), every=100)
print("average size of a first-layer weight:", round(decayed[0].weight.abs().mean().item(), 3),
      "with decay,", round(large[0].weight.abs().mean().item(), 3), "without")

epoch 100: training loss 0.589, validation loss 0.884
epoch 200: training loss 0.454, validation loss 0.906
epoch 300: training loss 0.356, validation loss 0.942
epoch 400: training loss 0.323, validation loss 0.965
lowest validation loss at epoch 85: 0.873
average size of a first-layer weight: 0.076 with decay, 0.271 without


Weight decay keeps the validation loss from climbing as far, to 0.965 by epoch 400 against 1.33
without it. Its lowest point, 0.873, is higher than the plain network's 0.84 in this run, a
difference one seed is too few to settle. The first layer's weights end at about a quarter of the
size, 0.076 against 0.271.

### Dropout and weight decay in Keras

`keras.layers.Dropout` and `keras.optimizers.AdamW` are the same tools. `large_keras` builds the
same network, and the cell fits it three ways, each on the 100 stations in one batch of 100.

In [8]:
def large_keras(dropout=0.0, optimizer=None):
    """The 4-128-128-1 network in Keras, compiled, from seed 0."""
    keras.utils.set_random_seed(0)
    layers = [keras.Input((4,)), keras.layers.Dense(128, activation="relu")]
    if dropout:
        layers.append(keras.layers.Dropout(dropout))
    layers.append(keras.layers.Dense(128, activation="relu"))
    if dropout:
        layers.append(keras.layers.Dropout(dropout))
    layers.append(keras.layers.Dense(1))
    model = keras.Sequential(layers)
    model.compile(optimizer=optimizer or keras.optimizers.Adam(0.01), loss="mse")
    return model


for name, model in [("no regularization", large_keras()),
                    ("dropout 0.3", large_keras(dropout=0.3)),
                    ("AdamW, decay 1.0", large_keras(optimizer=keras.optimizers.AdamW(0.01, weight_decay=1.0)))]:
    fitted = model.fit(scaled[few_rows], jan_mean[few_rows], validation_data=(scaled[valid_rows], jan_mean[valid_rows]),
                       epochs=400, batch_size=100, verbose=0)
    val_loss = fitted.history["val_loss"]
    best = int(np.argmin(val_loss))
    print(f"{name:17}: validation loss {val_loss[-1]:.3f} at epoch 400, lowest {val_loss[best]:.3f} at epoch {best + 1}")

no regularization: validation loss 1.479 at epoch 400, lowest 0.803 at epoch 67
dropout 0.3      : validation loss 0.843 at epoch 400, lowest 0.818 at epoch 148
AdamW, decay 1.0 : validation loss 0.924 at epoch 400, lowest 0.805 at epoch 83


Keras's network overfits the same way, from 0.803 at epoch 67 to 1.479 at epoch 400. Dropout and
weight decay hold the last epoch's validation loss to 0.843 and 0.924, and their lowest points,
0.818 and 0.805, are close to the plain network's. The three runs come from one seed and 200
validation stations, which is enough to show what each tool does to the curve and too little to
rank the tools.

### Early stopping by hand in PyTorch

The loop keeps going while the validation loss has improved within the last 50 epochs, keeps a deep
copy of the best weights, and loads them back at the end.

In [9]:
torch.manual_seed(0)
stopped = large_network()
optimizer = torch.optim.Adam(stopped.parameters(), lr=0.01)
best_loss, best_epoch, best_weights, epoch = float("inf"), 0, None, 0

while epoch - best_epoch < 50:
    epoch += 1
    stopped.train()
    optimizer.zero_grad()
    mse(stopped(X_few), y_few).backward()
    optimizer.step()
    stopped.eval()
    with torch.no_grad():
        val_loss = mse(stopped(X_valid), y_valid).item()
    if val_loss < best_loss:
        best_loss, best_epoch, best_weights = val_loss, epoch, copy.deepcopy(stopped.state_dict())

print(f"stopped after epoch {epoch}, with a validation loss of {val_loss:.3f}")
stopped.load_state_dict(best_weights)
with torch.no_grad():
    print(f"best epoch {best_epoch}: validation loss {mse(stopped(X_valid), y_valid).item():.3f}, "
          f"test loss {mse(stopped(X_test), y_test).item():.3f}")

stopped after epoch 126, with a validation loss of 0.940
best epoch 76: validation loss 0.840, test loss 1.055


Training ran 50 epochs past the best one and stopped after epoch 126, at a validation loss of 0.94.
Loading the copy put back epoch 76's weights, with that epoch's validation loss of 0.84. The test
loss, 1.055, is the one measure of this choice on stations that played no part in making it.

### Early stopping in Keras

`EarlyStopping` is a **callback**, an object whose methods `fit` calls at set points in training,
and this one acts at the end of every epoch. It watches `val_loss`, stops after 50 epochs without
improvement, and with `restore_best_weights=True` puts the best epoch's weights back.

In [10]:
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=50, restore_best_weights=True)
stopped_keras = large_keras()
fitted = stopped_keras.fit(scaled[few_rows], jan_mean[few_rows],
                           validation_data=(scaled[valid_rows], jan_mean[valid_rows]),
                           epochs=400, batch_size=100, verbose=0, callbacks=[early])

print("epochs run:", len(fitted.history["val_loss"]), "| best epoch:", early.best_epoch + 1)
print(f"validation loss now {stopped_keras.evaluate(scaled[valid_rows], jan_mean[valid_rows], verbose=0):.3f}, "
      f"test loss {stopped_keras.evaluate(scaled[test_rows], jan_mean[test_rows], verbose=0):.3f}")

epochs run: 117 | best epoch: 67
validation loss now 0.803, test loss 1.003


Keras stopped after 117 epochs, 50 past its best, epoch 67, and the restored model's validation
loss is that epoch's 0.803. `best_epoch` counts from 0, like `History.epoch`, so the cell adds 1.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/11-overfitting-and-regularization-solutions.ipynb).

**1.** Train the large network with a dropout rate of 0.5, from `torch.manual_seed(0)`, with `fit_full_batch`, and print its losses with `show`.

In [11]:
# your code here

**2.** Train the large network with `torch.optim.AdamW` at a weight decay of 0.1, and compare its lowest validation loss with the worked example's at a weight decay of 1.0.

In [12]:
# your code here

**3.** Run the early-stopping loop with a patience of 10 epochs, and print the epoch it stopped at, the best epoch and the validation loss of the weights it loads back.

In [13]:
# your code here

**4.** Give both hidden layers of a Keras version of the large network `kernel_regularizer=keras.regularizers.L2(0.001)`, compile it with `metrics=["mse"]`, fit it with an `EarlyStopping` like the worked example's that watches `val_mse`, and print the best epoch and that epoch's `val_mse`.

In [14]:
# your code here

**5.** Write a version of `fit_full_batch` that takes the training stations as arguments, train the large network without regularization on all 800 training stations, one full batch a step, for 400 epochs, and see whether its validation loss still turns.

In [15]:
# your code here

## Common errors

### No error, and a validation loss measured in training mode

The cell measures the dropout network's validation loss twice in training mode, the mode a training
step leaves it in, and then twice in evaluation mode.

In [16]:
torch.manual_seed(0)
dropout_net.train()
with torch.no_grad():
    print("training mode:  ", [round(mse(dropout_net(X_valid), y_valid).item(), 3) for _ in range(2)])
dropout_net.eval()
with torch.no_grad():
    print("evaluation mode:", [round(mse(dropout_net(X_valid), y_valid).item(), 3) for _ in range(2)])

training mode:   [1.236, 1.4]
evaluation mode: [0.877, 0.877]


In training mode the dropout layers switch off a fresh random choice of about 30% of the units on
every call, so the same stations get a different loss each time, and a worse one, since every
answer comes from a thinned network. Nothing warns about it. Call `model.eval()` before measuring, as `fit_full_batch`
does, and `model.train()` again before the next training step.

### No error, and the last weights rather than the best

This time `EarlyStopping` runs without `restore_best_weights`.

In [17]:
early_last = keras.callbacks.EarlyStopping(monitor="val_loss", patience=50)
last_keras = large_keras()
fitted = last_keras.fit(scaled[few_rows], jan_mean[few_rows],
                        validation_data=(scaled[valid_rows], jan_mean[valid_rows]),
                        epochs=400, batch_size=100, verbose=0, callbacks=[early_last])

print("epochs run:", len(fitted.history["val_loss"]), "| best epoch:", early_last.best_epoch + 1,
      f"| lowest validation loss {min(fitted.history['val_loss']):.3f}")
print(f"validation loss now {last_keras.evaluate(scaled[valid_rows], jan_mean[valid_rows], verbose=0):.3f}")

epochs run: 117 | best epoch: 67 | lowest validation loss 0.803
validation loss now 0.933


Training stopped at the right time, and the model kept the weights of the epoch it stopped at, 50
epochs past its best, with a validation loss of 0.933 against the best epoch's 0.803, since
`restore_best_weights` is `False` unless you set it.

### No error, and best weights that keep changing

The early-stopping loop with one change, `best_weights = model.state_dict()` in place of the deep
copy, run for 300 epochs.

In [18]:
torch.manual_seed(0)
model = large_network()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
best_loss, best_epoch, best_weights = float("inf"), 0, None
for epoch in range(1, 301):
    model.train()
    optimizer.zero_grad()
    mse(model(X_few), y_few).backward()
    optimizer.step()
    model.eval()
    with torch.no_grad():
        val_loss = mse(model(X_valid), y_valid).item()
    if val_loss < best_loss:
        best_loss, best_epoch, best_weights = val_loss, epoch, model.state_dict()

restored = large_network()
restored.load_state_dict(best_weights)
restored.eval()
with torch.no_grad():
    print(f"best epoch {best_epoch}, validation loss then {best_loss:.3f}")
    print(f"'best' weights loaded back: {mse(restored(X_valid), y_valid).item():.3f}")
    print(f"the model after epoch 300:  {val_loss:.3f}")
print("shares memory with the model's weights:", best_weights["0.weight"].data_ptr() == model[0].weight.data_ptr())

best epoch 76, validation loss then 0.840
'best' weights loaded back: 1.211
the model after epoch 300:  1.211
shares memory with the model's weights: True


`state_dict()` returns a dictionary of tensors that share memory with the model's weights, so every
later training step changes them too, in place. The dictionary saved at the best epoch therefore
holds epoch 300's weights by the end, and loading it gives epoch 300's validation loss, 1.211. Copy
it with `copy.deepcopy(model.state_dict())`, as the worked example does.

### `` UserWarning: Early stopping conditioned on metric `val_loss` which is not available. ``

Here `EarlyStopping` watches `val_loss` in a `fit` given no validation data.

In [19]:
blind = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", UserWarning)
    fitted = large_keras().fit(scaled[few_rows], jan_mean[few_rows], epochs=20, batch_size=100,
                               verbose=0, callbacks=[blind])

print(f"{caught[0].category.__name__}: {caught[0].message}")
print(len(caught), "warnings, and", len(fitted.history["loss"]), "epochs run")

watching = keras.callbacks.EarlyStopping(monitor="val_loss", patience=5)
with_data = large_keras().fit(scaled[few_rows], jan_mean[few_rows],
                              validation_data=(scaled[valid_rows], jan_mean[valid_rows]),
                              epochs=20, batch_size=100, verbose=0, callbacks=[watching])
print("with validation data:", len(with_data.history["loss"]), "epochs run")

20 warnings, and 20 epochs run
with validation data: 15 epochs run


Given the validation stations, the same callback stopped after 15 epochs. Without them the history
has no `val_loss`, so the callback warns at the end of every epoch and never stops, and all 20 epochs
ran. Give `fit` the validation data that the callback watches.

## Recap

- You can see overfitting in a record of training and validation losses, as the validation loss
  turns while the training loss keeps falling.
- You can add dropout to a network and switch it off for measuring with `model.eval()`.
- You can shrink weights with `AdamW`'s `weight_decay` in PyTorch and in Keras.
- You can stop training early by hand, keeping a deep copy of the best weights, and with Keras's
  `EarlyStopping` and `restore_best_weights=True`.

## What is next

The **Convolutional Networks** notebook moves from tables to images, where a layer that looks at
small patches of a picture needs far fewer weights than a dense one.

---

&#8592; **Previous:** [Training Loops](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/10-training-loops.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Convolutional Networks](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/12-convolutional-networks.ipynb) &#8594;
